In [ ]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns

# TensorFlow/Keras imports
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    from tensorflow.keras.optimizers import Adam
except ImportError:
    print("⚠️  TensorFlow not installed. Install with: pip install tensorflow")
    exit()

print("🧠 DEEP LEARNING MODEL TRAINING")
print("="*70)



In [ ]:
# Load data
X_train = np.load('data/X_train_scaled.npy')
X_test = np.load('data/X_test_scaled.npy')
y_train = np.load('data/y_train.npy')
y_test = np.load('data/y_test.npy')

print(f"✅ Training Set: {X_train.shape}")
print(f"✅ Testing Set: {X_test.shape}")
print(f"✅ Input Features: {X_train.shape[1]}")
print("="*70 + "\n")



In [ ]:
# Build Neural Network
print("🏗️  Building Neural Network Architecture...")

model = Sequential([
    # Input layer
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    
    # Hidden layers
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(16, activation='relu'),
    Dropout(0.2),
    
    # Output layer
    Dense(1, activation='sigmoid')
])



In [ ]:
# Compile model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

print("\n📐 MODEL ARCHITECTURE:")
model.summary()
print()

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)



In [ ]:
# Train model
print("🚀 Training Neural Network...")
print("="*70)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print("\n✅ Training Complete!")
print("="*70)



In [ ]:
# Evaluate model
print("\n📊 EVALUATING MODEL ON TEST SET...")
test_loss, test_accuracy, test_precision, test_recall = model.evaluate(X_test, y_test, verbose=0)

print(f"\n🎯 TEST SET PERFORMANCE:")
print("="*70)
print(f"  Loss: {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  Precision: {test_precision:.4f}")
print(f"  Recall: {test_recall:.4f}")
print(f"  F1-Score: {2 * (test_precision * test_recall) / (test_precision + test_recall):.4f}")
print("="*70)



In [ ]:
# Predictions
y_pred_proba = model.predict(X_test, verbose=0)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()



In [ ]:
# Classification Report
print("\n" + "="*70)
print("📋 CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred, target_names=['Low Rating', 'High Rating']))
print("="*70)



In [ ]:
# Save model
model.save('models/deep_learning_model.h5')
print("\n✅ Model saved: models/deep_learning_model.h5")

# Save training history
with open('models/dl_training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)
print("✅ Training history saved: models/dl_training_history.pkl")



In [ ]:
# VISUALIZATIONS
import os
os.makedirs('visualizations', exist_ok=True)

# 1. Training History
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[0, 0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[0, 0].set_title('Model Accuracy Over Epochs', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Loss
axes[0, 1].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 1].set_title('Model Loss Over Epochs', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Precision
axes[1, 0].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 0].plot(history.history['val_precision'], label='Validation Precision', linewidth=2)
axes[1, 0].set_title('Model Precision Over Epochs', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Recall
axes[1, 1].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 1].plot(history.history['val_recall'], label='Validation Recall', linewidth=2)
axes[1, 1].set_title('Model Recall Over Epochs', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualizations/13_dl_training_history.png', dpi=300, bbox_inches='tight')
print("✅ Saved: visualizations/13_dl_training_history.png")
plt.close()



In [ ]:
# 2. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label', fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')
ax.set_title('Confusion Matrix - Deep Learning Model', fontsize=14, fontweight='bold')
ax.set_xticklabels(['Low Rating', 'High Rating'])
ax.set_yticklabels(['Low Rating', 'High Rating'])
plt.tight_layout()
plt.savefig('visualizations/14_dl_confusion_matrix.png', dpi=300, bbox_inches='tight')
print("✅ Saved: visualizations/14_dl_confusion_matrix.png")
plt.close()



In [ ]:
# 3. ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontweight='bold')
ax.set_ylabel('True Positive Rate', fontweight='bold')
ax.set_title('ROC Curve - Deep Learning Model', fontsize=14, fontweight='bold')
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('visualizations/15_dl_roc_curve.png', dpi=300, bbox_inches='tight')
print("✅ Saved: visualizations/15_dl_roc_curve.png")
plt.close()

print("\n🎉 DEEP LEARNING MODEL TRAINING COMPLETE!")
print("="*70)